# Big Data y MLOps - Sesion 1
## Configuracion del entorno y extraccion de datos con PySpark

Bienvenido a la primera sesion del modulo de Big Data y MLOps. En las semanas anteriores trabajaste con Pandas para analizar datasets que caben comodamente en la memoria de tu computadora. A partir de hoy vamos a dar un salto: aprenderemos a trabajar con volumenes de datos que, en un escenario real, superan lo que una sola maquina puede procesar.

A lo largo de esta semana vas a construir un pipeline completo:

1. **Extraccion**: cargar datos masivos de forma distribuida con PySpark.
2. **Procesamiento**: limpiar, transformar y agregar esos datos con PySpark.
3. **Persistencia**: guardar los resultados en una base de datos relacional MySQL.

Al final del modulo vas a tener un pipeline funcional de extremo a extremo, que es exactamente el tipo de flujo de trabajo que se usa en la industria.

## Por que PySpark y no Pandas

Pandas procesa todo en la memoria RAM de una sola maquina. Funciona muy bien mientras el dataset quepa en esa memoria, pero quien trabaje con datos a nivel de terabytes o petabytes necesita otra herramienta.

PySpark es la interfaz de Python para Apache Spark, un motor de procesamiento que distribuye el trabajo entre varios nodos de un cluster. En lugar de que una sola maquina cargue todo el dataset, el trabajo se reparte entre varios "trabajadores" (workers), y cada uno procesa una porcion de los datos en paralelo.

Algunos casos reales donde esto se usa:

- Deteccion de fraude bancario en tiempo real.
- Sistemas de recomendacion masivos (Netflix, Amazon).
- Procesamiento de logs de servidores.

Dos buenas practicas que vas a ver aplicadas hoy y en la siguiente sesion:

- Preferir **DataFrames** sobre RDDs (Resilient Distributed Datasets), porque los DataFrames aprovechan el optimizador interno de Spark (Catalyst).
- Evitar funciones como `collect()` sobre datasets masivos, ya que intentan traer todos los datos distribuidos hacia la memoria local de una sola maquina, lo que puede causar errores de memoria.

```
Tamaño	Equivalencia	  Ejemplo
🟢 GB	1 GB ≈ 1,000 MB	  Dataset pequeño/mediano
🟡 TB	1 TB ≈ 1,000 GB	  Dataset muy grande
🔴 PB	1 PB ≈ 1,000 TB	  Dataset masivo

1 PB = 1,000 TB = 1,000,000 GB

## Instalacion de librerias

Para este modulo necesitas dos librerias principales:

- `pyspark`: para el procesamiento distribuido.
- `mysql-connector-python`: la vas a necesitar en la sesion mas adelante, pero es buena practica instalarla desde ahora.

Ejecuta la siguiente celda una sola vez. Si ya las tienes instaladas, no pasa nada al volver a correrla.

In [1]:
%pip install pyspark mysql-connector-python -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 61.1 MB/s eta 0:00:00


## Preparacion del dataset de trabajo

En una empresa real, este archivo vendria de un sistema de logs, un data warehouse o una exportacion de base de datos. Para poder practicar sin depender de una fuente externa, vamos a generar un dataset simulado que representa transacciones bancarias.

Este dataset va a tener las siguientes columnas:

- `id_transaccion`: identificador de la transaccion.
- `cliente_id`: identificador del cliente.
- `categoria`: categoria de la compra (Electronica, Ropa, Alimentos, Hogar, Deportes, Tecnologia).
- `monto`: monto de la transaccion.
- `canal`: canal por el que se hizo la compra (App, Web, Tienda Fisica).
- `fecha_hora`: fecha y hora de la transaccion.

A proposito, el dataset va a tener algunos problemas tipicos de datos del mundo real: valores nulos, texto con formato inconsistente (mayusculas y minusculas mezcladas), montos invalidos y algunos identificadores duplicados.

In [2]:
import numpy as np
import pandas as pd
import os

np.random.seed(42)

n_filas = 60000

categorias_base = ["Electronica", "Ropa", "Alimentos", "Hogar", "Deportes", "Tecnologia"]
canales = ["App", "Web", "Tienda Fisica"]

# Identificadores de transaccion, con algunos duplicados a proposito
id_transaccion = np.arange(1, n_filas + 1)
duplicados = np.random.choice(id_transaccion, size=200, replace=False)
id_transaccion = np.concatenate([id_transaccion, duplicados])
np.random.shuffle(id_transaccion)

n_total = len(id_transaccion)

cliente_id = np.random.randint(1000, 5000, size=n_total)

# Categoria con formato inconsistente (mayusculas, minusculas, mixto)
categoria_base_muestreada = np.random.choice(categorias_base, size=n_total)

def desordenar_texto(texto):
    opcion = np.random.random()
    if opcion < 0.3:
        return texto.upper()
    elif opcion < 0.6:
        return texto.lower()
    return texto

categoria = np.array([desordenar_texto(c) for c in categoria_base_muestreada], dtype=object)

# Monto con una distribucion realista (lognormal: muchos montos bajos, pocos muy altos)
monto = np.random.lognormal(mean=4.5, sigma=1.0, size=n_total).round(2).astype(object)

# Insertamos algunos montos negativos (errores de captura)
indices_monto_invalido = np.random.choice(n_total, size=150, replace=False)
for i in indices_monto_invalido:
    monto[i] = -monto[i]

# Insertamos valores nulos en monto y categoria
for i in np.random.choice(n_total, size=400, replace=False):
    monto[i] = None
for i in np.random.choice(n_total, size=300, replace=False):
    categoria[i] = None

canal = np.random.choice(canales, size=n_total, p=[0.5, 0.35, 0.15])

# Fechas distribuidas en un periodo de 60 dias
fecha_inicio = pd.Timestamp("2024-01-01")
offsets_minutos = np.random.randint(0, 60 * 24 * 60, size=n_total)
fecha_hora = fecha_inicio + pd.to_timedelta(offsets_minutos, unit="m")

df_transacciones = pd.DataFrame({
    "id_transaccion": id_transaccion,
    "cliente_id": cliente_id,
    "categoria": categoria,
    "monto": monto,
    "canal": canal,
    "fecha_hora": fecha_hora,
})

os.makedirs("data", exist_ok=True)
ruta_csv = "data/transacciones_masivas.csv"
df_transacciones.to_csv(ruta_csv, index=False)

print(f"Dataset generado con {len(df_transacciones)} filas.")
print(f"Guardado en: {ruta_csv}")


Dataset generado con 60200 filas.
Guardado en: data/transacciones_masivas.csv


In [3]:
df_transacciones.head(15)

,id_transaccion,cliente_id,categoria,monto,canal,fecha_hora
0,4680,2811,hogar,10.07,Web,2024-01-15 23:51:00
1,42232,1055,Hogar,72.17,App,2024-02-21 22:22:00
2,39843,4162,Deportes,6.3,Web,2024-01-06 00:27:00
3,27758,2322,Electronica,76.01,Web,2024-02-14 03:47:00
4,29348,4966,ALIMENTOS,113.44,Web,2024-02-12 11:56:00
5,14299,3491,Hogar,79.06,App,2024-02-11 15:08:00
6,38340,1561,Ropa,120.55,Tienda Fisica,2024-02-14 17:02:00
7,20798,1058,Electronica,16.34,Web,2024-01-15 04:02:00
8,51954,1083,HOGAR,47.93,App,2024-01-28 14:27:00
9,50932,3956,Alimentos,66.46,Web,2024-02-16 07:04:00


In [4]:
df_transacciones.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60200 entries, 0 to 60199
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id_transaccion  60200 non-null  int64         
 1   cliente_id      60200 non-null  int64         
 2   categoria       59900 non-null  object        
 3   monto           59800 non-null  object        
 4   canal           60200 non-null  object        
 5   fecha_hora      60200 non-null  datetime64[ns]
dtypes: datetime64[ns](1), int64(2), object(3)
memory usage: 2.8+ MB


## Creacion de la SparkSession

Para iniciar cualquier flujo de trabajo en PySpark, lo primero es configurar una `SparkSession`. Es el punto de entrada para todo lo que hagamos con Spark: leer archivos, crear DataFrames, ejecutar consultas, etc.

- `SparkSession.builder`: inicia el constructor de la sesion.
- `.appName(...)`: le da un nombre descriptivo a la tarea, util para identificarla en la interfaz de monitoreo de Spark.
- `.getOrCreate()`: recupera una sesion existente o crea una nueva si no hay ninguna activa.


--> **una SparkSession es lo que permite que nuestro notebook de Python se comunique con Spark.**

In [5]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder # Iniciamos el proceso para crear una SparkSession.
    .appName("MiApp") # Asignamos un nombre a nuestra aplicación de Spark.
    .getOrCreate() # Creamos la sesión o reutilizamos una que ya exista.
)

spark

## Carga del archivo CSV de forma distribuida

Ahora vamos a leer el CSV que generamos, pero esta vez usando Spark en lugar de Pandas. Dos parametros importantes:

- `header=True`: le indica a Spark que la primera fila contiene los nombres de las columnas.
- `inferSchema=True`: Spark analiza los datos para detectar automaticamente el tipo de cada columna (entero, decimal, texto, etc.).

Ten en cuenta que `inferSchema=True` obliga a Spark a leer el archivo una vez extra solo para deducir los tipos. Para un archivo de este tamano no es un problema, pero en archivos muy grandes esto puede ser lento.

In [6]:
ruta_csv = "data/transacciones_masivas.csv"

df_spark = spark.read.csv(ruta_csv, header=True, inferSchema=True)


## Primer vistazo a los datos

Antes de transformar cualquier dato, siempre conviene explorar la estructura del DataFrame:

- `printSchema()` muestra las columnas y los tipos de datos que Spark detecto.
- `show(n)` muestra las primeras `n` filas.
- `count()` cuenta el numero total de filas.

In [7]:
df_spark.printSchema()

df_spark.show(5)

print("Numero de filas:", df_spark.count())


root
 |-- id_transaccion: integer (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- categoria: string (nullable = true)
 |-- monto: double (nullable = true)
 |-- canal: string (nullable = true)
 |-- fecha_hora: timestamp (nullable = true)

+--------------+----------+-----------+------+-----+-------------------+
|id_transaccion|cliente_id|  categoria| monto|canal|         fecha_hora|
+--------------+----------+-----------+------+-----+-------------------+
|          4680|      2811|      hogar| 10.07|  Web|2024-01-15 23:51:00|
|         42232|      1055|      Hogar| 72.17|  App|2024-02-21 22:22:00|
|         39843|      4162|   Deportes|   6.3|  Web|2024-01-06 00:27:00|
|         27758|      2322|Electronica| 76.01|  Web|2024-02-14 03:47:00|
|         29348|      4966|  ALIMENTOS|113.44|  Web|2024-02-12 11:56:00|
+--------------+----------+-----------+------+-----+-------------------+
only showing top 5 rows
Numero de filas: 60200


```nullable = true``` → "Este field puede tener nullos."

## Seleccion, filtrado y Lazy Evaluation

Un concepto clave de Spark es la **Lazy Evaluation** (evaluacion perezosa). Cuando escribes una operacion como `select` o `filter`, Spark no la ejecuta de inmediato. En lugar de eso, construye un plan de ejecucion y solo lo ejecuta cuando le pides un resultado concreto, como `show()`, `count()` o `save()`.

Esto le permite a Spark optimizar el plan completo antes de mover cualquier dato, en lugar de ejecutar cada paso por separado.

In [8]:
df_filtrado = df_spark.select("cliente_id", "categoria", "monto") \
    .filter(df_spark["monto"] > 1000)

df_filtrado.show(10)


+----------+-----------+-------+
|cliente_id|  categoria|  monto|
+----------+-----------+-------+
|      2386|  Alimentos|1026.08|
|      4962| TECNOLOGIA|1559.77|
|      4726|  alimentos|1026.99|
|      2377|       Ropa|2959.45|
|      2255|Electronica|1400.23|
|      3227| TECNOLOGIA|1295.37|
|      4497|       ropa|1630.72|
|      1179|       ROPA|1135.93|
|      1536|   DEPORTES| 1225.5|
|      2797| Tecnologia| 1014.7|
+----------+-----------+-------+
only showing top 10 rows


la \ significa: "Esta instrucción todavía no terminó; continúa en la siguiente línea."  
Es equivalente a escribir:  
```
df_filtrado = (
    df_spark
    .select("cliente_id", "categoria", "monto")
    .filter(df_spark["monto"] > 1000))
```

En PySpark, es muy common encadenar varias operaciones:  

```
df \
    .filter(...) \
    .select(...) \
    .groupBy(...) \
    .count()
```

### Otro ejemplo de data manipulation

Digamos que queremos quedarse solo con las transacciones del canal `"App"` con un monto mayor a 500.

In [9]:
df_reto = df_spark.filter((df_spark["canal"] == "App") & (df_spark["monto"] > 500))
df_reto.show()
df_reto.count()



+--------------+----------+-----------+-------+-----+-------------------+
|id_transaccion|cliente_id|  categoria|  monto|canal|         fecha_hora|
+--------------+----------+-----------+-------+-----+-------------------+
|         56671|      3881|      hogar| 908.98|  App|2024-02-23 02:45:00|
|         10781|      1319|  alimentos| 938.96|  App|2024-01-08 21:57:00|
|         18862|      4962| TECNOLOGIA|1559.77|  App|2024-01-16 23:35:00|
|         53267|      4772|   DEPORTES|  599.1|  App|2024-02-15 03:26:00|
|         10119|      1333|       ROPA| 802.25|  App|2024-01-19 15:55:00|
|         15067|      4499| tecnologia| 722.26|  App|2024-02-05 12:26:00|
|         29683|      1585|      Hogar| 764.46|  App|2024-01-25 02:57:00|
|         24888|      3479|Electronica| 502.08|  App|2024-02-05 03:00:00|
|         18687|      3998|electronica| 630.19|  App|2024-01-27 23:07:00|
|         21777|      4159|       ROPA| 722.99|  App|2024-02-11 21:18:00|
|         41407|      2846|  Alimentos

1279

## ¿Por qué usamos PySpark en un solo notebook, si su idea es distribuir el trabajo entre varias maquinas?

Aca, Spark corre en modo local: en vez de usar varios nodos de un cluster, reparte el trabajo entre los cores de tu propia computadora. La API que usamos (SparkSession, DataFrames, groupBy, etc.) es exactamente la misma que se usaria en un cluster real, solo cambia la infraestructura por debajo. Por eso podemos aprender la logica distribuida sin necesitar un cluster: el mismo codigo, con cambios minimos, corre despues en produccion a mayor escala.

## Por dentro de Spark: driver, executors y particiones

La idea de esta seccion es abrir la "caja negra" de Spark: hasta ahora usaste `SparkSession`, `filter`, `groupBy`, etc. sin preguntarte que pasa por debajo cuando ejecutas esas operaciones.  
<br>
Entender esto te va a ayudar a razonar sobre rendimiento y a leer errores de Spark con mas contexto, tanto en este bootcamp como en un trabajo real con clusters grandes.  
<br>

Vamos a cubrir tres cosas:

1. El trio driver, cluster manager y executors.
2. Que pasa realmente cuando corremos Spark en modo local (como lo hemos hecho hasta ahora).
3. Que son las particiones y por que son la unidad basica de paralelismo en Spark.

## El trio: driver, cluster manager y executors

En un cluster real de Spark hay tres piezas:

- **Driver**: es el proceso principal. Ahi corre tu codigo de Python/Spark, ahi se construye el plan de ejecucion, y es quien coordina todo el trabajo. Piensa en el driver como el jefe de obra: no carga ladrillos, pero decide que se hace y en que orden.
- **Cluster manager**: es quien reparte los recursos disponibles (CPU, memoria) entre las distintas aplicaciones que se ejecutan en el cluster. Ejemplos: YARN, Kubernetes, o el propio "standalone manager" de Spark.
- **Executors**: son los procesos que realmente ejecutan las tareas y guardan datos en memoria o disco. Cada executor corre en una maquina del cluster, y puede ejecutar varias tareas en paralelo usando varios cores.

El flujo es asi: escribes una transformacion (`filter`, `groupBy`, etc.) en el driver, el driver arma un plan, el cluster manager le asigna executors, y esos executors ejecutan las tareas sobre los fragmentos de datos que les toco.

## Que pasa en modo local

Aca hay un detalle importante : en modo local **no existen executors separados como en un cluster real**. Todo el driver y el "executor" corren dentro del mismo proceso (la misma JVM=JavaVirtualMachine), en tu propia maquina.

Lo que si existe es paralelismo real: Spark usa varios **threads** (=tarea que puede ejecutarse en paralelo), uno por cada core que le permitas usar, y cada thread se comporta como si fuera un mini executor procesando una porcion de los datos. Por eso decimos que el modo local "simula" un cluster: el paralelismo es real, pero ocurre dentro de una sola maquina en lugar de repartirse entre varias.

Vamos a verlo en la practica.

In [10]:
# Ya tenemos una SparkSession creada ("spark", de la seccion anterior). La reutilizamos.

print("Master:", spark.sparkContext.master)  # Mostramos donde se esta ejecutando Spark.
print("Paralelismo por defecto:", spark.sparkContext.defaultParallelism)  # Cuantos threads puede usar Spark por defecto (puede variar con la maquina).


Master: local[*]
Paralelismo por defecto: 2


`local[*]` le dice a Spark: corre en modo local, y usa todos los cores disponibles en esta maquina.

`defaultParallelism` es la quantitad de tasks que Spark intenta ejecutar en paralelo por defecto. En modo local, normalmente coincide con el numero de cores de tu maquina. Podemos confirmarlo comparandolo con lo que reporta el sistema operativo.

In [11]:
import os

print("Cores segun el sistema operativo:", os.cpu_count())
print("Paralelismo por defecto de Spark:", spark.sparkContext.defaultParallelism)


Cores segun el sistema operativo: 2
Paralelismo por defecto de Spark: 2


> **Ques es un core (nucleo)?**  
> Un unidad de procesamiento dentro de la CPU (procesador) de una computadora, capaz de ejecutar instrucciones y realizar tareas de forma independiente.

## Que es una particion

Una **particion** es un fragmento del DataFrame. Spark no procesa todo el dataset de una sola vez: lo divide en particiones, y cada particion se procesa de forma independiente, potencialmente en paralelo.

Una analogia simple: si tienes que contar un mazo de 500 cartas entre 5 personas, le das 100 cartas a cada una (una particion por persona) y cada quien cuenta su parte al mismo tiempo. Al final sumas los resultados parciales. Eso es, a grandes rasgos, lo que hace Spark con las particiones.

Vamos a cargar el dataset que ya conoces y revisar cuantas particiones tiene.

In [12]:
# Ya tenemos "df_spark" cargado (de la seccion anterior). Revisamos cuantas particiones tiene.

print("Numero de particiones:", df_spark.rdd.getNumPartitions())


Numero de particiones: 1


Tener 1 partición significa que Spark ha cargado todo el CSV dataset en un único bloque de memoria que será procesado de forma secuencial por solo 1 core de la CPU, incluso si tu computadora o clúster tiene múltiples núcleos disponibles.

El numero de particiones es el limite de cuanto puede paralelizar Spark el trabajo:

- **Muy pocas particiones**: si tienes 1 sola particion, aunque tu maquina tenga 8 cores disponibles, Spark solo puede usar 1 a la vez para ese DataFrame. Estas desperdiciando paralelismo.
- **Demasiadas particiones**: si tienes miles de particiones muy pequenas, Spark gasta mas tiempo coordinando tareas diminutas que el que ahorra paralelizando. Cada tarea tiene un costo fijo de coordinacion.

```
[ 1 Partición ]   ---> Poco overhead, PERO 0 paralelismo (1 solo core trabaja).
[ 8 Particiones ]  ---> Equilibrio perfecto (todos los cores trabajan, casi no hay overhead).
[ 10,000 Partics ] ---> Muchísimo OVERHEAD (Spark pasa más tiempo administrando tareas que procesando los datos).
```

El objetivo es un numero de particiones que aproveche los cores disponibles sin generar overhead (sobrecarga) innecesario.

## Cambiando el numero de particiones: repartition vs coalesce

Spark ofrece dos formas de cambiar el numero de particiones de un DataFrame, y no son intercambiables:

| Metodo | Que hace | Costo |
|---|---|---|
| `repartition(n)` | Redistribuye los datos entre `n` particiones nuevas, moviendo datos entre ellas (shuffle) | Alto, pero balancea bien los datos |
| `coalesce(n)` | Combina particiones existentes para reducir el total, evitando mover datos innecesariamente | Bajo, pero solo sirve para reducir el numero de particiones, no para aumentarlo de forma balanceada |

En resumen: usa `repartition` cuando necesitas mas particiones o un balance parejo entre ellas, y `coalesce` cuando solo quieres reducir el numero de particiones de forma economica (por ejemplo, antes de escribir un archivo de salida).

In [13]:
# Redividir el DataFrame en 8 particiones
df_mas_particiones = df_spark.repartition(8)
print("Particiones despues de repartition(8):", df_mas_particiones.rdd.getNumPartitions())

# Reducir el número de particiones de 8 a 2
df_menos_particiones = df_mas_particiones.coalesce(2)
print("Particiones despues de coalesce(2):", df_menos_particiones.rdd.getNumPartitions())


Particiones despues de repartition(8): 8
Particiones despues de coalesce(2): 2


### En resumen (arquitectura)

Ahora que sabes que son el driver, los executors (o los threads que los simulan en modo local) y las particiones, las operaciones que ya conoces se leen distinto:

- Cuando defines un schema manual, le ahorras trabajo al driver.
- Cuando usas `groupBy`, estas provocando un shuffle que reparticiona los datos entre executors (o threads).
- Cuando el dataset es pequeno pero el numero de particiones es alto, estas pagando overhead de coordinacion sin necesidad.

Esta intuicion es la misma que vas a necesitar el dia que trabajes con un cluster real de varias maquinas: el codigo que escribes no cambia, pero ahora sabes que esta pasando por debajo.

## Cierre de la sesion

Hasta aqui ya tienes un DataFrame de Spark cargado a partir de un archivo masivo, y sabes como explorarlo y filtrarlo de forma basica. Sin embargo, el dataset todavia tiene nulos, texto inconsistente y valores invalidos.

En la siguiente sesion vamos a:

- Definir un schema manual para el DataFrame.
- Limpiar los valores nulos, duplicados e invalidos.
- Crear columnas calculadas.
- Calcular metricas agregadas con `groupBy`.

Guarda el archivo `data/transacciones_masivas.csv`, lo vamos a reutilizar en la proxima sesion.